# Liver cell GNN — Colab workspace

Use this notebook to:
1. Install dependencies
2. Set **your** data and repo paths
3. Run a **mock** sanity check (no real WSI required)

**Tip:** In Colab, use *Runtime → Change runtime type → GPU* for CUDA tests.

## 1) Environment installation

Run the cell below once per new runtime. Versions are pinned loosely for reproducibility; adjust if something conflicts with Colab’s default PyTorch.

In [ ]:
# check CUDA version
!nvcc --version
# check PyTorch version
!pip show torch
# check DGL version
!pip show dgl

In [ ]:
# CPU-only fallback: pip install dgl -f https://data.dgl.ai/wheels/repo.html

import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Core stack (add/remove as needed for your repo)
pip_install([
    "yacs",
    "lifelines",
    "scikit-learn",
    "pandas",
    "tqdm",
    "networkx",
    "Pillow",
    "opencv-python-headless",
])

# DGL: pick ONE line that matches your environment (see https://www.dgl.ai/pages/start.html)
# Example for CUDA 12.1 + torch 2.x (uncomment and fix URL if install fails):
# pip_install(["dgl", "-f", "https://data.dgl.ai/wheels/cu121/repo.html"])

# CPU wheel (safe default when CUDA match is unclear):
pip_install(["dgl", "-f", "https://data.dgl.ai/wheels/repo.html"])

print("pip installs finished.")

In [ ]:
# PyTorch Geometric & DeepSNAP — only if you import GraphLab loader end-to-end
# Uncomment when you need full training pipeline:
# !pip install -q torch-geometric
# !pip install -q deepsnap

print("Optional: uncomment PyG / DeepSNAP above when training.")

## 2) Google Drive (optional)

If your data and/or `liver_cell_gnn` live on Drive, mount and use paths like `/content/drive/MyDrive/...`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab — skip Drive mount.")

## 3) Paths — **fill in your values**

| Variable | Purpose |
|----------|--------|
| `REPO_ROOT` | Folder that contains the `GraphLab/` and `Run/` packages (parent of `liver_cell_gnn` if you keep that name) |
| `DATA_ROOT` | Root for datasets (graphs, CSVs, etc.) |
| `OUTPUT_ROOT` | Logs, checkpoints, notebook outputs |

Example (Colab + Drive): `REPO_ROOT = "/content/drive/MyDrive/projects/liver_cell_gnn"`

In [ ]:
from pathlib import Path

# --- EDIT THESE ---
REPO_ROOT = Path("/content/drive/MyDrive/path/to/liver_cell_gnn")  # contains GraphLab/, Run/
DATA_ROOT = Path("/content/drive/MyDrive/path/to/your_data")
OUTPUT_ROOT = Path("/content/drive/MyDrive/path/to/outputs")
# ------------------

for p in (DATA_ROOT, OUTPUT_ROOT):
    p.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT.resolve())
print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())

In [ ]:
import os
import sys

# So `import GraphLab` works when REPO_ROOT points at the repo root (same layout as local `Run/main.py`)
if REPO_ROOT.exists():
    sys.path.insert(0, str(REPO_ROOT))
    print("Added to sys.path:", REPO_ROOT)
else:
    print("WARNING: REPO_ROOT does not exist yet — clone/upload the repo or fix the path.")

## 4) Mock test run

Lightweight checks: **PyTorch**, **DGL**, optional **GraphLab** import. No WSI or `AllCell.bin` required.

If GraphLab fails to import, install missing deps from the error message or uncomment PyG/DeepSNAP above.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

x = torch.randn(4, 8, device="cuda" if torch.cuda.is_available() else "cpu")
y = x @ x.T
print("matmul shape:", y.shape, "ok")

In [ ]:
import dgl
import torch as th

g = dgl.graph(([0, 0, 1], [1, 2, 2]), num_nodes=3)
g.ndata["feat"] = th.randn(3, 16)
print("DGL graph:", g)
print("nodes:", g.num_nodes(), "edges:", g.num_edges())

In [ ]:
# Optional: import GraphLab (needs REPO_ROOT on sys.path and full dependency tree)
try:
    import GraphLab
    from GraphLab.config import cfg
    print("GraphLab import OK, cfg.seed =", cfg.seed)
except Exception as e:
    print("GraphLab import skipped or failed:", type(e).__name__, e)
    print("Install missing packages from the message, or run training only on a machine with the full env.")

In [ ]:
# Write a tiny marker file to OUTPUT_ROOT to confirm paths are writable
marker = OUTPUT_ROOT / "mock_run_ok.txt"
marker.write_text("mock run completed\n")
print("Wrote:", marker)

### Next steps

- Copy your **`Run/configs/IDGNN/graph.yaml`** and set `dataset.dir`, `out_dir`, etc. to paths under `DATA_ROOT` / `OUTPUT_ROOT`.
- From a terminal or `!cd` cell, run: `python main.py --cfg configs/IDGNN/graph.yaml` inside `Run/` once the full dataset is in place.
- See `liver_cell_gnn/README.md` → **Getting Started** for the on-disk layout (`train/val/test`, `AllCell.bin`).